## Universidade Cruzeiro do Sul
### Ciência de Dados e Aprendizado de Máquina
### 2º semestre de 2026

# Relatório do Projeto — NeuroWrite PT

**Projeto-terça:** trabalho acadêmico independente do EPA.

> **Versão de desenvolvimento.** Este notebook mantém a estrutura do modelo do professor. Ainda não há resultados empíricos porque o dataset próprio está em coleta. A versão de entrega deve ser executada e salva com todas as saídas.

## Grupo

- **Carlos** — desenvolvimento, processamento das imagens e experimentos.
- **Ian** — coleta manuscrita e documentação do dataset.
- **Isadora** — coleta manuscrita e análise das métricas.
- **Miguel** — coleta manuscrita e estudo dos classificadores.

*RGMs e nomes completos: preencher após confirmação dos integrantes.*

# NeuroWrite PT: reconhecimento de caracteres manuscritos em português

**Classificação de imagens de caracteres maiúsculos por meio de SVM e rede neural MLP.**

## Resumo

*A ser concluído após a execução e avaliação reais dos experimentos, incluindo quantidade final de imagens válidas, metodologia, melhores parâmetros, métricas, limitações e conclusões. Não preencher com resultados estimados.*

## Introdução

Escrever é uma atividade humana com variações de traço, proporções e espaçamento. O reconhecimento automático de caracteres manuscritos é um problema de **classificação supervisionada**: a cada imagem de uma letra, atribui-se uma classe conhecida. Após treinar com exemplos rotulados, um modelo tenta classificar imagens novas.

O NeuroWrite PT investiga especialmente a **generalização entre escritores**. A coleta envolve três pessoas, com imagens das 26 letras maiúsculas e 12 grafias adicionais acentuadas/com cedilha. Como complemento de literatura e possível comparação restrita a A–Z, consideramos o dataset público **EMNIST** (a variante escolhida deverá ser documentada antes do uso).

A classificação de caracteres **isolados** constitui o núcleo científico do projeto. A leitura de uma frase inteira dependeria adicionalmente de segmentação de palavras e letras, e é tratada como demonstração opcional, não como capacidade já comprovada.

## Objetivo

Comparar **SVM (Support Vector Machine)** e **MLP (Multilayer Perceptron)** na classificação de caracteres manuscritos em letras maiúsculas, investigando o desempenho quando os modelos recebem caligrafias de pessoas não utilizadas durante o treinamento.

**Objetivos específicos:**
1. Organizar uma base rotulada e reproduzível a partir da coleta manuscrita.
2. Documentar classes, dimensões, tipos de variáveis, eventuais perdas de amostras e distribuições.
3. Padronizar imagens, construir features de intensidade dos pixels e justificar essa representação.
4. Ajustar hiperparâmetros dos dois algoritmos e comparar métricas usando separação **por escritor**.
5. Interpretar erros e limitações, incluindo confusões em letras visualmente semelhantes.

**Hipótese:** a diferença entre caligrafias limita a generalização dos classificadores, mesmo com boa acurácia em exemplos semelhantes aos do treino.

## Desenvolvimento

### Parte 1 — Configuração, dados e análise exploratória

As folhas serão digitalizadas e os caracteres individualmente recortados e revisados. O arquivo `data/processed/manifest.csv` reunirá o caminho da imagem, o rótulo e o identificador do escritor. **Nesta fase, a segmentação automática não está implementada**: primeiro validaremos uma folha piloto de cada participante.

Todo o código da entrega será autocontido no notebook. Não haverá necessidade de abrir contas, montar Drive ou executar células durante a apresentação.

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, ConfusionMatrixDisplay

SEMENTE = 42
CLASSES = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ") + ["Ç", "Á", "À", "Â", "Ã", "É", "Ê", "Í", "Ó", "Ô", "Õ", "Ú"]
COLUNAS = ["arquivo", "classe", "escritor", "folha", "repeticao"]
CAMINHO_MANIFESTO = Path("data/processed/manifest.csv")

print(f"Classes planejadas: {len(CLASSES)}")
print(f"Amostras planejadas (3 escritores × 10 folhas × 2 repetições × 38 classes): {3*10*2*len(CLASSES)}")
print("Estes são números do protocolo de coleta, NÃO resultados do dataset.")

### Parte 1.1 — Dicionário e distribuição dos dados

O atributo de entrada será formado pelos **784 valores de intensidade** de uma imagem de 28 × 28 pixels, em escala de 0 a 1. A variável-alvo será a classe da letra (38 valores planejados). O identificador do escritor é usado apenas para agrupar a validação: **não pode entrar como atributo de treinamento**.

Ao concluir a segmentação, esta etapa permitirá exibir o número real de exemplos, a distribuição por classe e por escritor, valores ausentes e imagens de amostra.

In [ ]:
if CAMINHO_MANIFESTO.exists():
    manifesto = pd.read_csv(CAMINHO_MANIFESTO)
    faltantes = set(COLUNAS) - set(manifesto.columns)
    if faltantes:
        raise ValueError(f"Colunas obrigatórias ausentes no manifesto: {sorted(faltantes)}")
    print("Amostras reais carregadas:", len(manifesto))
    display(manifesto.head())
    fig, ax = plt.subplots(figsize=(12, 4))
    manifesto["classe"].value_counts().reindex(CLASSES, fill_value=0).plot.bar(ax=ax)
    ax.set(title="Quantidade de amostras por caractere", xlabel="Caractere", ylabel="Amostras")
    plt.tight_layout()
    plt.show()
else:
    manifesto = pd.DataFrame(columns=COLUNAS)
    print("Coleta ainda não importada. Esperado:", CAMINHO_MANIFESTO)
    print("Aguardando as fotos piloto e o manifesto das imagens recortadas.")

### Parte 1.2 — Normalização e vetor de características

Depois do recorte e inspeção de cada letra, converteremos as imagens para tons de cinza e 28 × 28 pixels. A intensidade do fundo claro se aproxima de 0, enquanto traços escuros passam a valores maiores pela inversão. Esta transformação pressupõe que **o recorte contenha um caractere isolado**; fotografias de páginas inteiras não devem ser enviadas diretamente a esta função.

A correção de rotação, enquadramento, pauta do caderno e preservação do acento será conferida com as páginas piloto.

In [ ]:
def vetorizar_caractere(caminho: str | Path, lado: int = 28) -> np.ndarray:
    """Converte um recorte individual com fundo claro em vetor de pixels [0,1]."""
    with Image.open(caminho) as arquivo:
        imagem = ImageOps.grayscale(arquivo)
        imagem = ImageOps.autocontrast(imagem)
        imagem = ImageOps.invert(imagem)
        imagem = imagem.resize((lado, lado), Image.Resampling.LANCZOS)
        vetor = np.asarray(imagem, dtype=np.float32).reshape(-1) / 255.0
    return vetor

if not manifesto.empty:
    exemplo = Path(str(manifesto.iloc[0]["arquivo"]))
    if exemplo.exists():
        plt.figure(figsize=(3, 3))
        plt.imshow(vetorizar_caractere(exemplo).reshape(28, 28), cmap="gray")
        plt.title("Exemplo real após normalização")
        plt.axis("off")
        plt.show()
    else:
        print("O manifesto existe, mas a imagem de exemplo ainda não está no caminho indicado.")
else:
    print("Amostra visual será exibida após recorte das fotos piloto.")

### Parte 2 — Implementação dos algoritmos

**Algoritmo A — SVM com kernel RBF.** Busca um limite de decisão que separa as classes no espaço das features. Testaremos `C` em 1 e 10.

**Algoritmo B — MLP.** Rede neural com camadas ocultas de 64 ou 128 neurônios. Testaremos as duas arquiteturas sob a mesma estratégia de avaliação.

Ambos utilizam `StandardScaler` dentro do `Pipeline`, evitando ajuste do escalonador sobre dados de teste. Usaremos busca de hiperparâmetros em validação interna.

#### Validação por escritor
Como o objetivo envolve generalizar para outra caligrafia, a separação de teste será **Leave-One-Group-Out**, com `escritor` como grupo. Cada rodada reserva integralmente um escritor para o teste, treina nos demais e calcula métricas; a busca de hiperparâmetros usa os escritores de treino, não o escritor de teste.

**Limitação:** três escritores produzem somente três rodadas externas e estimativas com alta incerteza. Não extrapolar esses resultados para a população geral. Caso o treino interno fique inviável, documentaremos a redução do grid sem consultar o teste.

In [ ]:
MODELOS = {
    "SVM": (
        Pipeline([
            ("escala", StandardScaler()),
            ("modelo", SVC(kernel="rbf"))
        ]),
        {"modelo__C": [1.0, 10.0]}
    ),
    "MLP": (
        Pipeline([
            ("escala", StandardScaler()),
            ("modelo", MLPClassifier(max_iter=200, early_stopping=False, random_state=SEMENTE))
        ]),
        {"modelo__hidden_layer_sizes": [(64,), (128,)]}
    ),
}

def avaliar_por_escritor(tabela: pd.DataFrame):
    """Validação externa por autor e busca interna por autor — sem vazamento entre escritores."""
    if tabela.empty:
        raise ValueError("Ainda não há caracteres no manifesto.")
    if tabela["escritor"].nunique() < 3:
        raise ValueError("É preciso coletar imagens de pelo menos 3 escritores distintos.")
    inexistentes = [s for s in tabela["arquivo"].astype(str) if not Path(s).exists()]
    if inexistentes:
        raise FileNotFoundError(f"{len(inexistentes)} imagens não encontradas. Ex.: {inexistentes[0]}")
    if tabela["classe"].isna().any():
        raise ValueError("Existem rótulos ausentes; revisar o manifesto.")
    X = np.vstack([vetorizar_caractere(p) for p in tabela["arquivo"].astype(str)])
    y = tabela["classe"].to_numpy()
    autores = tabela["escritor"].astype(str).to_numpy()
    externo = LeaveOneGroupOut()
    linhas = []
    predicoes = []

    for nome, (pipeline, parametros) in MODELOS.items():
        for rodada, (it, iv) in enumerate(externo.split(X, y, groups=autores), start=1):
            otim = GridSearchCV(
                estimator=pipeline, param_grid=parametros,
                scoring="f1_macro", cv=LeaveOneGroupOut(), n_jobs=1
            )
            otim.fit(X[it], y[it], groups=autores[it])
            previsto = otim.predict(X[iv])
            prec, rec, f1, _ = precision_recall_fscore_support(
                y[iv], previsto, average="macro", zero_division=0
            )
            autor_teste = ",".join(sorted(set(autores[iv])))
            linhas.append({
                "modelo": nome, "rodada": rodada, "escritor_teste": autor_teste,
                "acuracia": accuracy_score(y[iv], previsto),
                "precisao_macro": prec, "recall_macro": rec, "f1_macro": f1,
                "parametros": str(otim.best_params_)
            })
            predicoes.extend(zip([nome] * len(iv), [autor_teste] * len(iv), y[iv], previsto))
    return pd.DataFrame(linhas), pd.DataFrame(
        predicoes, columns=["modelo", "escritor_teste", "real", "previsto"]
    )

### Parte 3 — Execução, comparação e discussão dos resultados

Os resultados abaixo só serão calculados quando as fotografias forem segmentadas, conferidas e incluídas no manifesto. A comparação principal será **F1 macro**, acompanhada de acurácia, precisão macro, recall macro, diferenças entre escritores e matrizes de confusão.

Nunca avaliar a mesma amostra usada para escolher hiperparâmetros como se ela fosse um teste independente.

In [ ]:
if manifesto.empty:
    print("Experimento não executado: dataset manuscrito ainda não disponibilizado.")
    print("Próximo passo: validar o recorte de uma folha piloto por escritor.")
else:
    resultados, previsoes = avaliar_por_escritor(manifesto)
    display(resultados)
    display(resultados.groupby("modelo")[["acuracia", "precisao_macro", "recall_macro", "f1_macro"]].agg(["mean", "std"]).round(3))
    for nome in MODELOS:
        sub = previsoes[previsoes["modelo"] == nome]
        cm = confusion_matrix(sub["real"], sub["previsto"], labels=CLASSES)
        fig, ax = plt.subplots(figsize=(12, 10))
        ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(ax=ax, xticks_rotation=90, cmap="Blues", values_format="d", colorbar=False)
        ax.set_title(f"Matriz de confusão — {nome} (escritores não vistos)")
        plt.tight_layout()
        plt.show()

### Parte 4 — Teste de texto manuscrito (complementar)

A classificação de caracteres isolados **não resolve automaticamente** o reconhecimento de frases. Antes de demonstrar a leitura de texto, precisaremos:
- isolar corretamente caracteres em imagem de frase inédita;
- identificar espaços e preservar acentos/cedilha;
- aplicar o classificador já escolhido **sem retreiná-lo na frase de teste**;
- apresentar também os erros e limitações, se ocorrerem.

Este bloco só será implementado se houver tempo após concluir os requisitos obrigatórios de classificação.

## Conclusão

*Pendente de evidência empírica.* Após os experimentos, registrar o melhor modelo de acordo com as métricas, os melhores hiperparâmetros, comportamento por escritor, principais erros, limites do pequeno conjunto de pessoas e contribuição real da metodologia.

Nenhuma conclusão sobre acurácia deve ser redigida antes da execução.

## Referências

- COHEN, G. et al. **EMNIST: an extension of MNIST to handwritten letters.** 2017. Fonte e variantes: https://www.nist.gov/itl/products-and-services/emnist-dataset
- SCIKIT-LEARN. **SVC.** https://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
- SCIKIT-LEARN. **MLPClassifier.** https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html
- SCIKIT-LEARN. **LeaveOneGroupOut.** https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.LeaveOneGroupOut.html
- SCIKIT-LEARN. **Model evaluation.** https://scikit-learn.org/stable/modules/model_evaluation.html

*Completar a bibliografia conforme as fontes efetivamente consultadas no trabalho.*